# 02 Transform Silver

Builds cleaned, typed views on top of Bronze: `silver_subscriptions`, `silver_payments`, `silver_chargebacks`.

- Casts strings to dates, timestamps and doubles. Blank `end_date` and `resolved_ts` become NULL.
- Adds `month` (payments) and `month_opened` (chargebacks) as `yyyy-MM` strings for the Gold rollups.
- `DEDUP_STRATEGY=keyed` (default) keeps one row per natural key:
  - payments: one row per `payment_id` (latest `ts`). Removes ingest replays.
  - chargebacks: one row per (`payment_id`, `opened_ts`), preferring the resolved row over an earlier pending one.
- `DEDUP_STRATEGY=none` passes duplicates through, so you can see their effect on Gold.
- Quality checks at the end fail the run on NULL keys, or on duplicate keys when deduping.

In [1]:
dbutils.widgets.text("CATALOG", "")
dbutils.widgets.text("SCHEMA", "demo_finance")
dbutils.widgets.dropdown("DEDUP_STRATEGY", "keyed", ["none", "keyed"])

CATALOG = dbutils.widgets.get("CATALOG").strip() or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("SCHEMA").strip()
DEDUP_STRATEGY = dbutils.widgets.get("DEDUP_STRATEGY").strip().lower()
if not SCHEMA:
    raise ValueError("SCHEMA must not be empty")
if DEDUP_STRATEGY not in {"none", "keyed"}:
    raise ValueError(f"DEDUP_STRATEGY must be 'none' or 'keyed', got {DEDUP_STRATEGY!r}")

spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")

missing = [t for t in ["bronze_subscriptions", "bronze_payments", "bronze_chargebacks"]
           if not spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{t}")]
if missing:
    raise RuntimeError(f"Missing Bronze tables {missing} in {CATALOG}.{SCHEMA}. Run 01_ingest_bronze first.")
print(f"schema: {CATALOG}.{SCHEMA}  dedup: {DEDUP_STRATEGY}")

schema: <catalog>.demo_finance_runnable  dedup: keyed


In [2]:
# Subscriptions: one row per customer in the source, so no dedupe needed.
spark.sql("""
CREATE OR REPLACE VIEW silver_subscriptions AS
SELECT
  CAST(customer_id AS STRING)                      AS customer_id,
  CAST(plan        AS STRING)                      AS plan,
  TO_DATE(start_date)                              AS start_date,
  CASE WHEN end_date IS NULL OR end_date = '' THEN NULL ELSE TO_DATE(end_date) END AS end_date,
  CAST(amount      AS DOUBLE)                      AS amount,
  CAST(status      AS STRING)                      AS status
FROM bronze_subscriptions
""")
print("created view silver_subscriptions")

created view silver_subscriptions


In [3]:
# Payments. Natural key: payment_id. Keyed dedupe keeps the latest row by ts.
payments_typed = """
  SELECT
    CAST(payment_id  AS STRING)  AS payment_id,
    CAST(customer_id AS STRING)  AS customer_id,
    CAST(amount      AS DOUBLE)  AS amount,
    CAST(status      AS STRING)  AS status,
    CAST(method      AS STRING)  AS method,
    to_timestamp(ts)             AS ts,
    date_format(to_timestamp(ts), 'yyyy-MM') AS month
  FROM bronze_payments
"""
if DEDUP_STRATEGY == "keyed":
    spark.sql(f"""
    CREATE OR REPLACE VIEW silver_payments AS
    WITH typed AS ({payments_typed}),
    ranked AS (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY ts DESC, amount DESC) AS rn
      FROM typed
    )
    SELECT payment_id, customer_id, amount, status, method, ts, month
    FROM ranked
    WHERE rn = 1
    """)
else:
    spark.sql(f"CREATE OR REPLACE VIEW silver_payments AS {payments_typed}")
print(f"created view silver_payments (dedup={DEDUP_STRATEGY})")

created view silver_payments (dedup=keyed)


In [4]:
# Chargebacks. Natural key: (payment_id, opened_ts). Keyed dedupe keeps the
# resolved row when a case arrived first as pending and later as resolved.
chargebacks_typed = """
  SELECT
    CAST(payment_id AS STRING)   AS payment_id,
    CAST(reason     AS STRING)   AS reason,
    CAST(outcome    AS STRING)   AS outcome,
    to_timestamp(opened_ts)      AS opened_ts,
    to_timestamp(resolved_ts)    AS resolved_ts,
    date_format(to_timestamp(opened_ts), 'yyyy-MM') AS month_opened
  FROM bronze_chargebacks
"""
if DEDUP_STRATEGY == "keyed":
    spark.sql(f"""
    CREATE OR REPLACE VIEW silver_chargebacks AS
    WITH typed AS ({chargebacks_typed}),
    ranked AS (
      SELECT *,
             ROW_NUMBER() OVER (
               PARTITION BY payment_id, opened_ts
               ORDER BY resolved_ts DESC NULLS LAST
             ) AS rn
      FROM typed
    )
    SELECT payment_id, reason, outcome, opened_ts, resolved_ts, month_opened
    FROM ranked
    WHERE rn = 1
    """)
else:
    spark.sql(f"CREATE OR REPLACE VIEW silver_chargebacks AS {chargebacks_typed}")
print(f"created view silver_chargebacks (dedup={DEDUP_STRATEGY})")

created view silver_chargebacks (dedup=keyed)


In [5]:
# Quality checks: row counts, rows removed by dedupe, NULL keys, duplicate keys.
def scalar(sql):
    return spark.sql(sql).first()[0]

print(f"{'view':<22}{'bronze':>8}{'silver':>8}{'removed':>9}")
for name in ["subscriptions", "payments", "chargebacks"]:
    b = spark.table(f"bronze_{name}").count()
    s = spark.table(f"silver_{name}").count()
    print(f"silver_{name:<15}{b:>8}{s:>8}{b - s:>9}")

checks = {
    "NULL customer_id in silver_subscriptions": "SELECT COUNT(*) FROM silver_subscriptions WHERE customer_id IS NULL",
    "NULL payment_id in silver_payments":       "SELECT COUNT(*) FROM silver_payments WHERE payment_id IS NULL",
    "NULL payment_id in silver_chargebacks":    "SELECT COUNT(*) FROM silver_chargebacks WHERE payment_id IS NULL",
    "NULL ts in silver_payments":               "SELECT COUNT(*) FROM silver_payments WHERE ts IS NULL",
    "duplicate customer_id in silver_subscriptions":
        "SELECT COUNT(*) FROM (SELECT customer_id FROM silver_subscriptions GROUP BY customer_id HAVING COUNT(*) > 1)",
    "duplicate payment_id in silver_payments":
        "SELECT COUNT(*) FROM (SELECT payment_id FROM silver_payments GROUP BY payment_id HAVING COUNT(*) > 1)",
    "duplicate (payment_id, opened_ts) in silver_chargebacks":
        "SELECT COUNT(*) FROM (SELECT payment_id, opened_ts FROM silver_chargebacks GROUP BY payment_id, opened_ts HAVING COUNT(*) > 1)",
}
failures = []
for label, sql in checks.items():
    n = scalar(sql)
    blocking = not label.startswith("duplicate") or DEDUP_STRATEGY == "keyed"
    flag = "ok" if n == 0 else ("FAIL" if blocking else "warn")
    print(f"[{flag:>4}] {label}: {n}")
    if n and blocking:
        failures.append(label)
if failures:
    raise AssertionError(f"Silver quality checks failed: {failures}")

print("Payment status mix:", {r[0]: r[1] for r in spark.sql("SELECT status, COUNT(*) FROM silver_payments GROUP BY status ORDER BY status").collect()})
print("Open chargebacks (no resolved_ts):", scalar("SELECT COUNT(*) FROM silver_chargebacks WHERE resolved_ts IS NULL"))
print("Silver transform complete.")

view                    bronze  silver  removed
silver_subscriptions      1000    1000        0
silver_payments          12722   12621      101
silver_chargebacks         161     156        5
[  ok] NULL customer_id in silver_subscriptions: 0
[  ok] NULL payment_id in silver_payments: 0
[  ok] NULL payment_id in silver_chargebacks: 0
[  ok] NULL ts in silver_payments: 0
[  ok] duplicate customer_id in silver_subscriptions: 0
[  ok] duplicate payment_id in silver_payments: 0
[  ok] duplicate (payment_id, opened_ts) in silver_chargebacks: 0
Payment status mix: {'failed': 912, 'pending': 49, 'settled': 11660}
Open chargebacks (no resolved_ts): 23
Silver transform complete.
